## Import

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:1'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [3]:
from datasets.Student_t import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 2
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.00
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [4]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.K_components = 6
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [5]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 6 copula transform= True
nde type: FM
finished: t= 0 loss= 3.618610382080078 loss val= 1.9241188764572144 best val loss= 1.9241188764572144 best t= 0
finished: t= 126 loss= 1.1433535814285278 loss val= 1.2699037790298462 best val loss= 1.1818439960479736 best t= 86
finished: t= 252 loss= 1.1076838970184326 loss val= 1.3181381225585938 best val loss= 1.1818439960479736 best t= 86


finished: t= 0 loss= 3.2805886268615723 loss val= 1.9661587476730347 best val loss= 1.9661587476730347 best t= 0
finished: t= 126 loss= 0.9737022519111633 loss val= 1.3170485496520996 best val loss= 1.266528606414795 best t= 83
finished: t= 252 loss= 1.0816558599472046 loss val= 1.403529405593872 best val loss= 1.2618582248687744 best t= 131


finished: t= 0 loss= 449.88702392578125 loss val= 441.584228515625 best val loss= 441.584228515625 best t= 0
finished: t= 101 loss= 78.37708282470703 loss val= 79.11962890625 best val loss= 79.10734558105469 best t= 98
finished: t= 202 loss= 78.31640625 lo

In [8]:
n = len(X)

n_array = [50, 100, 500, 1000, 2500, 5000]
var_array = []
for n_data in n_array:
    var = []
    for i in range(8):
        idx = torch.randperm(n)
        XX, YY = X[idx][0:n_data], Y[idx][0:n_data]
        mi = estimator.MI(XX, YY)
        var.append(mi)
    var = np.array(var)
    var_array.append(var.std())
    print('n:', n_data, 'std', var.std())

n: 50 std 0.6242474366899008
n: 100 std 0.4764094771901249
n: 500 std 0.18140770114455224
n: 1000 std 0.1786813820119302
n: 2500 std 0.050677679419090585
n: 5000 std 0.034235989189828495
